# African Healthcare Indicators
## Python / Anaconda Classwork – 10 Questions

**Dataset:** `africa_healthcare.csv` | 140 rows | 10 selected African countries | 2010–2023

**Libraries:** pandas, matplotlib, seaborn

**Instructions:** Complete all questions and explain your findings. Questions 6–10 require five charts. Give every chart a title, axis labels and units; use a legend where appropriate.

**Countries:** Nigeria, Ghana, Kenya, South Africa, Egypt, Ethiopia, Rwanda, Uganda, Tanzania and Senegal.

**Source:** World Bank World Development Indicators (retrieved 28 October 2026).


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Load the dataset
# Change the path if the CSV is in a different location
df = pd.read_csv("africa_healthcare.csv")
print("Dataset loaded successfully.")


## Question 1 – Load and inspect the data

Load the CSV into pandas. Display the first 10 rows, the number of rows and columns, and the data types.  
How many countries and years are represented?


In [ ]:
# First 10 rows
print("First 10 rows:")
display(df.head(10))

# Shape
print(f"\nNumber of rows and columns: {df.shape}")

# Data types
print("\nData types:")
print(df.dtypes)

# Unique countries and years
n_countries = df['country'].nunique()
n_years = df['year'].nunique()
print(f"\nNumber of countries: {n_countries}")
print(f"Number of years: {n_years}")
print(f"Years represented: {sorted(df['year'].unique())}")
print(f"\nCountries:\n{df['country'].unique()}")


**Findings**  
- The dataset has **140 rows** and **8 columns**.  
- There are **10 countries** and **14 years** (2010–2023 inclusive).  
- Each row represents one country in one year.  
- Data types are appropriate: strings for country identifiers, `int64` for year and DPT %, `float64` for the continuous health indicators.


## Question 2 – Missing values and duplicates

Check for missing values and duplicate country–year records. Report the results and explain how you would handle missing values if present.  
**Do not replace missing health measurements with zero.**


In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Duplicate country-year pairs
duplicates = df.duplicated(subset=['country', 'year'], keep=False)
print(f"\nNumber of duplicate country–year records: {duplicates.sum()}")
if duplicates.any():
    display(df[duplicates])
else:
    print("No duplicate country–year records found.")


**Findings**  
- There are **no missing values** in any column.  
- There are **no duplicate country–year records**.

**How I would handle missing values (if present)**  
1. Report the extent and pattern of missingness (by country and by variable).  
2. For analysis, prefer list-wise deletion or exclusion of NaNs so that summary statistics and correlations are computed only on observed data.  
3. Country-specific linear interpolation or multiple imputation could be considered if the missingness is limited and the time series is otherwise complete.  
4. **Never replace missing health measurements with zero** — this would create severe downward bias in means, distort rankings, and produce misleading correlations.


## Question 3 – 2023 summary statistics

For 2023, calculate the mean, median, minimum and maximum of life expectancy, under-five mortality and DPT immunization across the selected countries. Treat each country equally.


In [ ]:
df2023 = df[df['year'] == 2023].copy()

cols = ['life_expectancy_years', 'under5_mortality_per_1000', 'dpt_immunization_pct']
summary = df2023[cols].agg(['mean', 'median', 'min', 'max']).round(2)
print("2023 Summary Statistics (unweighted, equal weight per country):")
display(summary)

print("\nCountry-level values for reference:")
display(df2023[['country'] + cols].sort_values('life_expectancy_years', ascending=False))


**Findings (2023, unweighted)**  

| Statistic | Life expectancy (years) | Under-5 mortality (per 1 000) | DPT immunization (%) |
|-----------|--------------------------|-------------------------------|----------------------|
| Mean      | 66.04                    | 46.77                         | 86.0                 |
| Median    | 67.16                    | 38.65                         | 91.5                 |
| Min       | 54.46 (Nigeria)          | 22.9 (Egypt)                  | 67 (Nigeria)         |
| Max       | 71.63 (Egypt)            | 116.8 (Nigeria)               | 96 (Egypt)           |

The median life expectancy and DPT coverage are higher than the means, while median under-five mortality is lower than the mean. This reflects right-skew in mortality (driven largely by Nigeria) and left-skew in immunization coverage.


## Question 4 – Nigeria 2010 vs 2023

Filter the data for Nigeria.  
- By how many years did life expectancy change between 2010 and 2023?  
- Calculate the percentage reduction in under-five mortality over the same period:  
  `(2010 value − 2023 value) / 2010 value × 100`


In [ ]:
nga = df[df['country'] == 'Nigeria'].sort_values('year')

le_2010 = nga.loc[nga['year'] == 2010, 'life_expectancy_years'].values[0]
le_2023 = nga.loc[nga['year'] == 2023, 'life_expectancy_years'].values[0]
le_change = le_2023 - le_2010

u5_2010 = nga.loc[nga['year'] == 2010, 'under5_mortality_per_1000'].values[0]
u5_2023 = nga.loc[nga['year'] == 2023, 'under5_mortality_per_1000'].values[0]
u5_reduction_pct = (u5_2010 - u5_2023) / u5_2010 * 100

print("Nigeria key indicators:")
display(nga[['year', 'life_expectancy_years', 'under5_mortality_per_1000']])

print(f"\nLife expectancy change (2010 → 2023): {le_change:+.2f} years")
print(f"Under-five mortality reduction: {u5_reduction_pct:.2f}%")
print(f"  (from {u5_2010} to {u5_2023} deaths per 1,000 live births)")


**Findings**  
- Life expectancy in Nigeria increased by **+3.15 years** (from 51.32 to 54.46 years).  
- Under-five mortality declined by **7.52%** (from 126.3 to 116.8 deaths per 1,000 live births).  

Progress is positive but modest compared with several peer countries in the sample.


## Question 5 – Countries with DPT immunization below 90% (2023)

Using 2023 data, list countries with DPT immunization below 90%.  
Show their immunization and under-five mortality values.  
Use 90% as a classroom comparison threshold, not a clinical recommendation.


In [ ]:
low_dpt = (df2023[df2023['dpt_immunization_pct'] < 90]
           [['country', 'dpt_immunization_pct', 'under5_mortality_per_1000']]
           .sort_values('dpt_immunization_pct'))

print("Countries with DPT immunization < 90% in 2023:")
display(low_dpt)
print(f"\nNumber of countries below 90%: {len(low_dpt)}")


**Findings**  
Four countries had DPT immunization coverage below 90% in 2023:

| Country      | DPT % | Under-5 mortality (per 1 000) |
|--------------|-------|-------------------------------|
| Nigeria      | 67    | 116.8                         |
| Ethiopia     | 70    | 51.5                          |
| South Africa | 79    | 34.8                          |
| Tanzania     | 83    | 38.5                          |

Nigeria combines the lowest coverage with by far the highest under-five mortality in the sample.


## Question 6 – Bar chart: Under-five mortality by country (2023)

Compare under-five mortality across all 10 countries in 2023.  
Sort from highest to lowest and label the axis as deaths per 1,000 live births.  
Which three countries have the highest values?


In [ ]:
data6 = df2023.sort_values('under5_mortality_per_1000', ascending=False)

plt.figure(figsize=(11, 6))
ax = sns.barplot(data=data6, x='country', y='under5_mortality_per_1000',
                 color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_title('Under-five Mortality Rate by Country, 2023', fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Deaths per 1,000 live births')
plt.xticks(rotation=45, ha='right')
# Add value labels on bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print("Top 3 countries (highest under-five mortality):")
print(data6[['country', 'under5_mortality_per_1000']].head(3).to_string(index=False))


**Interpretation**  
The three countries with the highest under-five mortality in 2023 are:  
1. **Nigeria** – 116.8  
2. **Ethiopia** – 51.5  
3. **Uganda** – 49.4  

Nigeria is a clear outlier, with mortality more than twice as high as the next country.


## Question 7 – Line chart: Life expectancy trends (2010–2023)

Plot life expectancy from 2010 to 2023 for Nigeria, Ghana, Kenya and South Africa. Add a legend.  
Which country has the largest absolute increase over the period?


In [ ]:
selected = ['Nigeria', 'Ghana', 'Kenya', 'South Africa']
df_sel = df[df['country'].isin(selected)]

plt.figure(figsize=(11, 6))
for country in selected:
    sub = df_sel[df_sel['country'] == country]
    plt.plot(sub['year'], sub['life_expectancy_years'], marker='o', linewidth=2, label=country)

plt.title('Life Expectancy at Birth, 2010–2023\n(Nigeria, Ghana, Kenya, South Africa)', fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Life expectancy (years)')
plt.legend(title='Country')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate absolute increases
print("Absolute change in life expectancy (2010 → 2023):")
for c in selected:
    sub = df[df['country'] == c]
    le0 = sub.loc[sub['year'] == 2010, 'life_expectancy_years'].values[0]
    le1 = sub.loc[sub['year'] == 2023, 'life_expectancy_years'].values[0]
    print(f"  {c:15s}: {le1 - le0:+.2f} years  ({le0:.2f} → {le1:.2f})")


**Interpretation**  
**South Africa** recorded the largest absolute increase in life expectancy (+7.25 years), followed by Ghana (+4.20), Nigeria (+3.15) and Kenya (+2.73).  

South Africa’s strong recovery is consistent with improved HIV treatment access and broader health-system gains after the early-2000s peak of the epidemic.


## Question 8 – Horizontal bar chart: DPT immunization (2023)

Compare DPT immunization coverage across the 10 countries in 2023.  
Add a vertical reference line at 90%.  
How many countries meet or exceed this threshold?


In [ ]:
data8 = df2023.sort_values('dpt_immunization_pct')

plt.figure(figsize=(10, 7))
bars = plt.barh(data8['country'], data8['dpt_immunization_pct'],
                color='teal', edgecolor='black', linewidth=0.5)
plt.axvline(90, color='crimson', linestyle='--', linewidth=2, label='90% threshold')
plt.title('DPT Immunization Coverage by Country, 2023', fontweight='bold')
plt.xlabel('DPT immunization (% of children ages 12–23 months)')
plt.ylabel('Country')
plt.legend(loc='lower right')
# Value labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.5, bar.get_y() + bar.get_height()/2,
             f'{width:.0f}%', va='center', fontsize=9)
plt.xlim(0, 105)
plt.tight_layout()
plt.show()

n_meet = (df2023['dpt_immunization_pct'] >= 90).sum()
print(f"Number of countries meeting or exceeding 90%: {n_meet}")
print("\nCountries at or above 90%:")
print(df2023.loc[df2023['dpt_immunization_pct'] >= 90,
                 ['country', 'dpt_immunization_pct']]
      .sort_values('dpt_immunization_pct', ascending=False)
      .to_string(index=False))


**Interpretation**  
**6 countries** meet or exceed the 90% classroom threshold:  
Egypt (96%), Ghana (95%), Rwanda (94%), Kenya (93%), Senegal (92%) and Uganda (91%).  

The four countries below the threshold are Nigeria (67%), Ethiopia (70%), South Africa (79%) and Tanzania (83%).


## Question 9 – Scatter plot: Health expenditure per capita vs life expectancy (2023)

For 2023, plot health expenditure per capita (current US$) on the x-axis and life expectancy on the y-axis.  
Label each country and calculate Pearson correlation.  
Describe the association and explain why it does not prove causation.


In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df2023,
                x='health_expenditure_per_capita_usd',
                y='life_expectancy_years',
                s=100, color='darkblue', edgecolor='black')

for _, row in df2023.iterrows():
    label = row['country'].replace(', Arab Rep.', '')
    plt.annotate(label,
                 (row['health_expenditure_per_capita_usd'], row['life_expectancy_years']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8)

plt.title('Health Expenditure per Capita vs Life Expectancy, 2023', fontweight='bold')
plt.xlabel('Current health expenditure per capita (current US$)')
plt.ylabel('Life expectancy at birth (years)')
plt.tight_layout()
plt.show()

r, pval = pearsonr(df2023['health_expenditure_per_capita_usd'],
                   df2023['life_expectancy_years'])
print(f"Pearson correlation coefficient (r) = {r:.3f}")
print(f"p-value = {pval:.3f}")

print("\nData points:")
display(df2023[['country', 'health_expenditure_per_capita_usd', 'life_expectancy_years']]
        .sort_values('health_expenditure_per_capita_usd'))


**Interpretation**  
The Pearson correlation is approximately **r = 0.047** (very weak / essentially none).  

South Africa spends by far the most per capita yet has only mid-range life expectancy, while several lower-spending countries (Egypt, Senegal, Uganda, Rwanda) achieve higher life expectancy.  

**Why this does not prove causation**  
- The analysis is a single-year cross-section of only 10 countries.  
- Many important confounders (HIV prevalence, conflict, governance quality, education, sanitation, income inequality, disease burden) are not controlled for.  
- Reverse causality and measurement error in expenditure data are possible.  
- Correlation alone cannot establish that higher spending *causes* longer life expectancy.


## Question 10 – Heatmap: Pearson correlation matrix (2023)

Using 2023 data only, show the Pearson correlation matrix for the five health indicators. Annotate the values.  
Which pair has the strongest absolute off-diagonal correlation?  
Explain one limitation of drawing conclusions from only 10 countries.


In [ ]:
indicators = [
    'life_expectancy_years',
    'under5_mortality_per_1000',
    'dpt_immunization_pct',
    'health_expenditure_pct_gdp',
    'health_expenditure_per_capita_usd'
]

corr = df2023[indicators].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.6, cbar_kws={'shrink': 0.8})
plt.title('Pearson Correlation Matrix of Health Indicators, 2023', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Correlation matrix:")
display(corr.round(3))

# Strongest absolute off-diagonal
pairs = []
for i, c1 in enumerate(indicators):
    for j, c2 in enumerate(indicators):
        if i < j:
            pairs.append(((c1, c2), abs(corr.loc[c1, c2]), corr.loc[c1, c2]))

pairs_sorted = sorted(pairs, key=lambda x: -x[1])
print("\nStrongest absolute off-diagonal correlations:")
for (c1, c2), abs_r, r in pairs_sorted[:3]:
    print(f"  {c1}  ↔  {c2}:  r = {r:+.3f}  (|r| = {abs_r:.3f})")


**Interpretation**  
The pair with the **strongest absolute off-diagonal correlation** is:  

**`health_expenditure_pct_gdp` ↔ `health_expenditure_per_capita_usd`**  
(r ≈ **+0.919**)

(The next strongest is life expectancy ↔ under-five mortality, r ≈ –0.898.)

**Limitation of drawing conclusions from only 10 countries**  
With n = 10 the sample is extremely small. Correlations are unstable, confidence intervals are wide, and a single outlier (e.g., Nigeria or South Africa) can dominate the result. The selected countries are also not a random or representative sample of Africa, so generalising the observed associations to the whole continent (or claiming causal relationships) is not justified.


---
## Submission checklist

- [x] Completed notebook with visible outputs and calculations  
- [x] Five required charts (Questions 6–10) with titles, axis labels (including units) and legends where appropriate  
- [x] Short interpretations for each question  
- [x] Rates are never summed  
- [x] Unweighted country averages are clearly distinguished from any population-weighted estimates (none were used)

**Notes on data**  
- Values are World Bank estimates and may be revised.  
- Current US$ figures are not adjusted for inflation or purchasing power.  
- This selected-country sample does not represent all of Africa.
